# Lab 19 · Prognoza wielodniowa

Przed wykonaniem tego laboratorium przeczytaj wykład [D15.4 „Odróżnij direct od recursive forecasting”](../lectures/d15-multistep.html).

Pamiętaj, że do tego labu potrzebne jest środowisko z labu 9: projekt practice z bibliotekami sekcji C. Uruchom Jupyter tak samo jak tam.

Pod opisem każdego zadania znajduje się komórka kodu do uzupełnienia, a pod nią komórka sprawdzenia. Jeżeli zadanie nie jest jeszcze rozwiązane, sprawdzenie zatrzyma się i wyświetli czerwony komunikat. To normalne: komunikat mówi, co trzeba poprawić.

Najpierw przewidź wynik, potem uruchom kod. Podpowiedzi otwieraj pojedynczo.

In [ ]:
from pathlib import Path
import sys
assert sys.version_info[:2] == (3, 12), "Wybierz kernel Pythona 3.12 z głównego README."
lab_dir = Path.cwd()
assert (lab_dir / "lab-19-prognoza-wielodniowa.ipynb").is_file(), "Uruchom Jupyter z folderu labs."
print("Folder:", lab_dir)

In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError as error:
    raise AssertionError("Brakuje bibliotek sekcji C. Dodaj je w practice i uruchom Jupyter przez --project ../practice, zgodnie z głównym README.") from error
print("pandas", pd.__version__, "| numpy", np.__version__)

## Zamrożona chwila prognozy

Gotowa komórka odtwarza `train` i `valid` oraz model jednodniowy z C12: `one_day_model` dopasowany na `lag1`. Chwila prognozy to wieczór 11 lutego 2024, ostatni dzień treningu. `last_known` to sprzedaż z tego dnia, a `horizon_days` to trzy kolejne daty. Sprzedaż z tych dni jest w tabeli, ale w chwili prognozy jej nie znasz.

In [ ]:
sales = pd.read_csv(lab_dir / "data" / "project-development.csv", parse_dates=["day"])
assert sales.shape == (49, 2), "Oczekiwano 49 dni i dwóch kolumn."
cutoff = pd.Timestamp("2024-02-12")
with_lag = sales.assign(lag1=sales["sale"].shift(1))
rows = with_lag[with_lag["lag1"].notna()]
train = rows[rows["day"] < cutoff]
valid = rows[rows["day"] >= cutoff]
assert len(train) == 41 and len(valid) == 7
print("Trening:", train["day"].min().date(), "–", train["day"].max().date(), "| walidacja:", valid["day"].min().date(), "–", valid["day"].max().date())
from sklearn.linear_model import LinearRegression
one_day_model = LinearRegression().fit(train[["lag1"]], train["sale"])
origin = train["day"].max()
last_known = float(train.loc[train["day"] == origin, "sale"].item())
horizon_days = valid["day"].iloc[:3].tolist()
print("Chwila prognozy:", origin.date(), "| ostatnia znana sprzedaż:", last_known)
print("Horyzont:", [day.date().isoformat() for day in horizon_days])

## D15.4 · Recursive: prognoza zastępuje brakującą historię

Po wykładzie D15.4 wykonaj trzy kroki rekurencyjne modelem `one_day_model`. W `recursive_inputs` zapisz listę wartości `lag1` podanych modelowi w kolejnych krokach, a w `recursive_forecast` listę trzech prognoz. Pierwsze wejście to `last_known`; każde kolejne to poprzednia prognoza, nie rzeczywista sprzedaż. Do predict podaj `pd.DataFrame({"lag1": [wartość]})`, żeby zachować nazwę cechy.

In [ ]:
recursive_inputs = None
recursive_forecast = None

In [ ]:
assert isinstance(recursive_inputs, list) and isinstance(recursive_forecast, list) and len(recursive_inputs) == len(recursive_forecast) == 3, "D15.4: trzy wejścia i trzy prognozy."
assert np.isclose(recursive_inputs[0], last_known), "D15.4: pierwszy krok używa ostatniej znanej sprzedaży."
assert np.allclose(recursive_inputs[1:], recursive_forecast[:2]), "D15.4: drugie i trzecie wejście to poprzednie prognozy, nie sprzedaż z tabeli."
assert np.allclose(recursive_forecast, [17.105646667914332, 17.098317429382046, 17.097106297434635]), "D15.4: każda prognoza pochodzi z one_day_model i bieżącego wejścia."
print("D15.4: ślad recursive:", [round(v, 3) for v in recursive_inputs], "->", [round(v, 3) for v in recursive_forecast])

<details><summary>Podpowiedź 1</summary>

Pętla po trzech krokach: zapisz wejście, policz prognozę, uczyń ją wejściem następnego kroku.

</details>

<details><summary>Podpowiedź 2</summary>

Tabela valid nie bierze udziału w tym zadaniu; jej użycie zmieniłoby protokół na codziennie aktualizowany.

</details>

## D15.4 · Direct: osobny model na każdy krok

Dla kroku `h` równego 1, 2 i 3 utwórz z `sales` cechę `lag_h` przez `shift(h)`, zostaw wiersze ze znaną cechą i dniem przed `cutoff`, dopasuj nową `LinearRegression` i policz prognozę na dzień `origin + h` z wejściem `last_known`. W `direct_inputs` zapisz listę trzech wejść, a w `direct_forecast` listę trzech prognoz. Przewidź, czy wejścia będą się różnić między krokami.

In [ ]:
direct_inputs = None
direct_forecast = None

In [ ]:
assert isinstance(direct_inputs, list) and isinstance(direct_forecast, list) and len(direct_inputs) == len(direct_forecast) == 3, "D15.4: trzy wejścia i trzy prognozy."
assert np.allclose(direct_inputs, [last_known] * 3), "D15.4: w direct każdy krok używa tylko informacji z chwili prognozy, czyli last_known."
assert np.allclose(direct_forecast, [17.105646667914332, 17.164944678198534, 17.262643675794102]), "D15.4: osobny model na każdy krok, z cechą shift(h) i treningiem przed cutoff."
print("D15.4: ślad direct:", [round(v, 3) for v in direct_inputs], "->", [round(v, 3) for v in direct_forecast])

<details><summary>Podpowiedź 1</summary>

Pętla po h in [1, 2, 3]: assign z shift(h), maska notna, maska day < cutoff, fit, predict.

</details>

<details><summary>Podpowiedź 2</summary>

Nazwa kolumny cechy może być ta sama w każdym obiegu; liczy się przesunięcie h.

</details>

## D15.4 · Ten sam horyzont, trzy protokoły

W `actual_3` są rzeczywiste sprzedaże trzech dni horyzontu. W `recursive_errors` i `direct_errors` zapisz listy wielkości błędów obu prognoz dzień po dniu. W `one_day_errors` zapisz listę błędów modelu `one_day_model` z rzeczywistym `lag1` z `valid`, czyli protokołu codziennie aktualizowanego z C12. To trzecia procedura z innym dostępem do informacji; nie łącz jej wyników z zamrożonym horyzontem.

In [ ]:
actual_3 = valid["sale"].iloc[:3].tolist()
recursive_errors = None
direct_errors = None
one_day_errors = None

In [ ]:
for name, errors, wanted in [("recursive_errors", recursive_errors, [0.19435333208566874, 5.351682570617953, 2.502893702565366]), ("direct_errors", direct_errors, [0.19435333208566874, 5.285055321801465, 2.3373563242058992]), ("one_day_errors", one_day_errors, [0.19435333208566874, 5.319566338005508, 1.618546207920044])]:
    assert isinstance(errors, list) and len(errors) == 3 and all(e >= 0 for e in errors), f"D15.4: {name} to trzy nieujemne wielkości błędów."
    assert np.allclose(errors, wanted), f"D15.4: {name}: porównaj prognozy z actual_3 dzień po dniu; one_day używa rzeczywistego lag1 z valid."
print("D15.4: recursive", [round(e, 2) for e in recursive_errors], "| direct", [round(e, 2) for e in direct_errors], "| codziennie aktualizowany", [round(e, 2) for e in one_day_errors])

<details><summary>Podpowiedź 1</summary>

abs różnicy actual_3 i prognozy, element po elemencie.

</details>

<details><summary>Podpowiedź 2</summary>

one_day_model.predict(valid[["lag1"]].iloc[:3]) daje prognozy z rzeczywistym wczoraj.

</details>

## Sprawdź rezultat

Sprawdzenia przechodzą po restarcie kernela. Do przemyślenia: dlaczego błąd drugiego dnia w recursive zależy od pierwszej prognozy? Czy niższy błąd protokołu codziennie aktualizowanego dowodzi lepszego modelu? Który zapis trafiłby do pliku wyników jako osobny eksperyment z datą przygotowania? Nie musisz spisywać odpowiedzi.